In [ ]:
import numpy as np
import pandas as pd
from astropy import units as u
from astropy.coordinates import SkyCoord
from astroquery.vizier import Vizier
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

TARGET_RA,TARGET_DEC = 8.9721766,43.5615651	
SEARCH_RADIUS = 5.0 * u.arcsec

CATALOGS = {
    'NOMAD1': 'I/297/out',
    'GSC 2.3': 'I/305/out',
    'PPMXL': 'I/317/sample',
    'TIC v8.2': 'IV/39/tic82',
    'ATLAS Refcat2': 'J/ApJ/867/105/refcat2',
    
    'Gaia DR3': 'I/355/gaiadr3',
    
    'GALEX AIS': 'II/335/galex_ais',
    'Swift UVOT': 'II/311/uvot',
    'XMM-Newton OM': 'II/312/xmm_om',
    'ASTROSAT UVIT': 'II/352/uvit',
    'IUE Final Archive': 'II/215/catalog',
    
    'SDSS DR16': 'V/154/sdss16',
    'Pan-STARRS DR1': 'II/349/ps1',
    
    '2MASS': 'II/246/out',
    'AllWISE': 'II/328/allwise',
}

WAVELENGTHS = {
    'FUV': 1516.0, 'NUV': 2267.0,
    'U': 3600.0, 'UVW1': 2910.0, 'UVW2': 2090.0, 'UVM2': 2246.0,
    'u': 3551.0, 'g': 4686.0, 'r': 6165.0, 'i': 7481.0, 'z': 8931.0, 'y': 9620.0,
    'J': 12350.0, 'H': 16620.0, 'Ks': 21590.0, 'K': 21590.0,
    'W1': 33526.0, 'W2': 46028.0, 'W3': 115608.0, 'W4': 220883.0,
    'G': 6730.0, 'Gbp': 5320.0, 'Grp': 7970.0,
    'B': 4450.0, 'V': 5510.0, 'R': 6580.0, 'I': 8060.0,
}

def extract_obs_time(row):
    time_keywords = [
        'jd', 'mjd', 'epoch', 'obsdate', 't_obs', 'date_obs', 'dateobs',
        't_start', 't_end', 't_mean', 'obsjd', 'obsmjd', 'obstime'
    ]
    
    for col in row.colnames:
        col_lower = col.lower()
        if any(kw in col_lower for kw in time_keywords):
            try:
                val = row[col]
                if not np.ma.is_masked(val):
                    fval = float(val)
                    if fval > 1000:
                        return fval
            except:
                pass
    return None

def check_sdss_spec(ra, dec, radius):
    print("Checking SDSS spectroscopy...")
    try:
        coord = SkyCoord(ra=ra, dec=dec, unit=(u.deg, u.deg), frame='icrs')
        v = Vizier(row_limit=5, columns=['*'])
        result = v.query_region(coord, radius=radius, catalog='V/147/sdss12')
        
        if len(result) > 0:
            table = result[0]
            if '_r' in table.columns:
                table.sort('_r')
            row = table[0]
            
            spec_info = {
                'has_spec': True,
                'z': row['z'] if 'z' in row.colnames else None,
                'class': row['class'] if 'class' in row.colnames else None,
            }
            print(f"  [FOUND] SDSS Spectrum: z={spec_info['z']}, class={spec_info['class']}")
            return spec_info
    except:
        pass
    
    print("  [--] No SDSS Spectrum")
    return {'has_spec': False}

def query_vizier_photometry(ra, dec, radius):
    print(f"Querying Vizier (RA={ra:.4f}, DEC={dec:.4f})...\n")
    
    coord = SkyCoord(ra=ra, dec=dec, unit=(u.deg, u.deg), frame='icrs')
    v = Vizier(row_limit=5, columns=['*'])
    
    phot_data = []
    obs_times = {}

    for cat_name, vizier_id in CATALOGS.items():
        try:
            result = v.query_region(coord, radius=radius, catalog=vizier_id)
            
            if len(result) > 0:
                table = result[0]
                if '_r' in table.columns:
                    table.sort('_r')
                row = table[0]
                
                obs_time = extract_obs_time(row)
                if obs_time:
                    obs_times[cat_name] = obs_time
                
                for col in row.colnames:
                    if 'mag' in col.lower() and 'e_' not in col and not np.ma.is_masked(row[col]):
                        mag_val = row[col]
                        
                        band = None
                        if 'FUV' in col: band = 'FUV'
                        elif 'NUV' in col: band = 'NUV'
                        elif 'UVW1' in col: band = 'UVW1'
                        elif 'UVW2' in col: band = 'UVW2'
                        elif 'UVM2' in col: band = 'UVM2'
                        elif 'Umag' in col or col == 'U': band = 'U'
                        elif 'umag' in col or col == 'u': band = 'u'
                        elif 'gmag' in col or col == 'g': band = 'g'
                        elif 'rmag' in col or col == 'r': band = 'r'
                        elif 'imag' in col or col == 'i': band = 'i'
                        elif 'zmag' in col or col == 'z': band = 'z'
                        elif 'ymag' in col or col == 'y': band = 'y'
                        elif 'Jmag' in col or col == 'J': band = 'J'
                        elif 'Hmag' in col or col == 'H': band = 'H'
                        elif 'Kmag' in col or 'Ksmag' in col: band = 'Ks'
                        elif 'W1' in col: band = 'W1'
                        elif 'W2' in col: band = 'W2'
                        elif 'W3' in col: band = 'W3'
                        elif 'W4' in col: band = 'W4'
                        elif 'Gmag' in col: band = 'G'
                        elif 'BP' in col: band = 'Gbp'
                        elif 'RP' in col: band = 'Grp'
                        
                        if band and band in WAVELENGTHS:
                            phot_data.append({
                                'Catalog': cat_name,
                                'Band': band,
                                'Mag': float(f"{mag_val:.3f}"),
                                'Wave_A': WAVELENGTHS[band],
                                'ObsTime': obs_time if obs_time else np.nan,
                            })
                
                print(f"  [OK] {cat_name:25s}", end='')
                if cat_name in obs_times:
                    print(f" JD={obs_times[cat_name]:.4f}")
                else:
                    print()
            else:
                print(f"  [--] {cat_name:25s}")

        except Exception as e:
            print(f"  [xx] {cat_name:25s}")
            
    return pd.DataFrame(phot_data), obs_times

def calculate_flux(df):
    if df.empty or 'Mag' not in df.columns:
        return df
        
    ZP_JY = {
        'FUV': 3631, 'NUV': 3631, 'U': 1760, 'UVW1': 1620, 'UVW2': 2090, 'UVM2': 2246,
        'u': 3631, 'g': 3631, 'r': 3631, 'i': 3631, 'z': 3631, 'y': 9620.0,
        'J': 1594, 'H': 1024, 'Ks': 666, 'K': 666,
        'W1': 309, 'W2': 171, 'W3': 31, 'W4': 8,
        'G': 3228, 'Gbp': 3552, 'Grp': 2555,
        'B': 4063, 'V': 3640, 'R': 3080, 'I': 2550,
    }

    fluxes = []
    for idx, row in df.iterrows():
        band = row['Band']
        mag = row['Mag']
        wave = row['Wave_A']
        
        zp = ZP_JY.get(band, 3631)
        f_nu_jy = zp * 10**(-0.4 * mag)
        c_ang = 2.998e18
        f_lambda = (f_nu_jy * 1e-23) * c_ang / (wave**2)
        fluxes.append(f_lambda)
        
    df['Flux_cgs'] = fluxes
    return df

if __name__ == "__main__":
    print("="*80)
    print("MULTI-WAVELENGTH PHOTOMETRY")
    print("="*80 + "\n")
    
    spec_info = check_sdss_spec(TARGET_RA, TARGET_DEC, SEARCH_RADIUS)
    print()
    
    df_phot, obs_times = query_vizier_photometry(TARGET_RA, TARGET_DEC, SEARCH_RADIUS)
    
    if not df_phot.empty:
        df_phot = df_phot.drop_duplicates(subset=['Band', 'Wave_A', 'Mag'], keep='first').reset_index(drop=True)
        df_phot = calculate_flux(df_phot)
        df_phot_sorted = df_phot.sort_values('Wave_A')
        
        print("\n" + "="*80)
        print(df_phot_sorted[['Catalog', 'Band', 'Wave_A', 'Mag', 'Flux_cgs', 'ObsTime']].to_string(index=False))
        
        if obs_times:
            first_time = list(obs_times.values())[0]
            obs_time_str = f"JD{first_time:.0f}"
        else:
            obs_time_str = datetime.now().strftime('%Y%m%d')
        
        output_file = f"Photometry_RA{TARGET_RA:.2f}_DEC{TARGET_DEC:.2f}.csv"
        df_phot_sorted.to_csv(output_file, index=False)
        
        print(f"\n{'='*80}")
        print(f"Saved: {output_file}")
        print(f"Total: {len(df_phot_sorted)} | UV: {len(df_phot_sorted[df_phot_sorted['Wave_A'] < 4000])} | Opt: {len(df_phot_sorted[(df_phot_sorted['Wave_A'] >= 4000) & (df_phot_sorted['Wave_A'] < 12000)])} | IR: {len(df_phot_sorted[df_phot_sorted['Wave_A'] >= 12000])}")
        if spec_info['has_spec']:
            print(f"SDSS Spectroscopy: YES")
    else:
        print("No data found.")

In [ ]:
import pandas as pd
import os
from astroquery.sdss import SDSS
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.io import fits
import warnings

warnings.filterwarnings('ignore')

# ================= 配置 =================

INPUT_FILE = 'apjad7500t1_mrt.csv'  # 输入文件
OUTPUT_DIR = 'SDSS_Spectra_Downloads'      # 光谱保存目录
SEARCH_RADIUS = 3.0                       

# 如果目录不存在，自动创建
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ================= 工具函数 =================

def clean_filename(name):
    """清理文件名中的非法字符 (空格, /, :)"""
    return str(name).replace(' ', '_').replace('/', '-').replace(':', '').strip()

def process_target(row):
    """处理单个目标：查询 -> 下载 -> 保存"""
    
    # 1. 获取基本信息
    try:
        identifier = row['Candidate']
        ra = float(row['RAdeg'])
        dec = float(row['DEdeg'])
    except KeyError as e:
        print(f"❌ [错误] CSV 缺少列: {e}")
        return
    except ValueError:
        print(f"❌ [错误] 坐标格式无效: {identifier}")
        return

    safe_name = clean_filename(identifier)
    
    # 2. 定义坐标
    pos = SkyCoord(ra, dec, unit=(u.deg, u.deg), frame='icrs')
    
    # 3. 查询 SDSS (spectro=True 仅搜索有光谱的数据)
    print(f"🔍 正在查询: {identifier} (RA={ra:.4f}, DEC={dec:.4f}) ... ", end="", flush=True)
    
    try:
        # query_region 返回一个 Table
        xid = SDSS.query_region(pos, radius=SEARCH_RADIUS * u.arcsec, spectro=True)
    except Exception as e:
        print(f"网络/查询错误")
        return

    # 4. 检查结果
    if xid is None or len(xid) == 0:
        print("无 SDSS 光谱")
        return
    
    print(f"✅ 发现 {len(xid)} 条光谱! 正在下载...")

    # 5. 下载数据
    try:
        spectra = SDSS.get_spectra(matches=xid)
    except Exception as e:
        print(f"  ❌ 下载过程中断: {e}")
        return

    # 6. 保存文件
    # 我们遍历查询结果 xid 和下载结果 spectra
    for i, sp in enumerate(spectra):
        # 获取 SDSS 的唯一标识符
        plate = xid[i]['plate']
        mjd = xid[i]['mjd']
        fiber = xid[i]['fiberID']
        
        # 构造文件名: Identifier_spec-PLATE-MJD-FIBER.fits
        # 这样既保留了你的名字，也保留了 SDSS 官方编号
        filename = f"{safe_name}_spec-{plate}-{mjd}-{fiber:04d}.fits"
        save_path = os.path.join(OUTPUT_DIR, filename)
        
        try:
            sp.writeto(save_path, overwrite=True)
            print(f"   -> 已保存: {filename}")
        except Exception as e:
            print(f"   -> 保存失败: {e}")

# ================= 主程序 =================
def main():
    if not os.path.exists(INPUT_FILE):
        print(f"错误: 找不到输入文件 {INPUT_FILE}")
        return

    print(f"正在读取 {INPUT_FILE} ...")
    df = pd.read_csv(INPUT_FILE)
    
    total = len(df)
    print(f"开始处理 {total} 个目标...\n")

    for index, row in df.iterrows():
        print(f"[{index+1}/{total}] ", end="")
        process_target(row)

    print("\n" + "="*30)
    print(f"全部完成。光谱文件已保存在 '{OUTPUT_DIR}' 文件夹中。")

if __name__ == "__main__":
    main()

In [ ]:
画ztf的图像，要求画的过程中要# 设置为您想查看的频率范围。您提到在 10 左右有一个峰。
# 我们限制搜索范围在 5 到 15 之间，这样代码就会忽略左边的低频峰，只看右边。
SEARCH_FREQ_MIN = 5.0   
SEARCH_FREQ_MAX = 15.0  
画2个图，一个是最大周期的，另一个是第二高的峰

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import P4J
import os
import wget
from astropy.coordinates import SkyCoord
import astropy.units as u
import warnings
import sys

warnings.filterwarnings('ignore')

# ================= 配置 =================
INPUT_FILE = 'PolarCat_Full_Combined_gaia.csv' # 输入文件
DATA_DIR = 'LC_Data_ZTF'                   # ZTF CSV 下载存放目录
OUTPUT_PLOT_DIR = 'LC_Plots_ZTF'           # 图片保存路径

# 如果文件夹不存在，自动创建
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)
if not os.path.exists(OUTPUT_PLOT_DIR):
    os.makedirs(OUTPUT_PLOT_DIR)

# ================= 工具函数 =================

def sanitize_filename(name):
    """清理文件名，去除非法字符"""
    return str(name).replace(' ', '_').replace('/', '-').replace(':', '').strip()

def fold(time, period):
    return np.mod(time, period)/period % 1.0      

def AOV(t, y, dy):
    """执行 P4J 周期搜索"""
    try:
        my_per = P4J.periodogram(method='MHAOV') 
        my_per.set_data(t, y, dy)
        
        # 频率搜索范围 (1/1000天 ~ 145次/天)
        my_per.frequency_grid_evaluation(fmin=1e-3, fmax=145.0, fresolution=1e-4)
        my_per.finetune_best_frequencies(fresolution=1e-5, n_local_optima=10)
        
        freq, per = my_per.get_periodogram()
        fbest, pbest = my_per.get_best_frequencies()
        
        objperiod = 1.0 / fbest[0]
        # print(f'  Best Period: {objperiod:.6f} d ({objperiod*24*60:.2f} min)')
        
        return objperiod, freq, per, fbest, pbest
    except Exception as e:
        print(f"  [P4J Error] {e}")
        return None, None, None, None, None

def download_ztf(ra, dec, filepath):
    """下载 ZTF 数据"""
    # 0.00083 deg ≈ 3 arcsec
    url = f"https://irsa.ipac.caltech.edu/cgi-bin/ZTF/nph_light_curves?POS=CIRCLE+{ra}+{dec}+0.00083&COLLECTION=&FORMAT=csv"
    try:
        # print(f"  Downloading from: {url}")
        wget.download(url, out=filepath, bar=None) # bar=None 不显示进度条，避免刷屏
        return True
    except Exception as e:
        print(f"  [下载失败] {e}")
        return False

def process_target(row, ra_col, dec_col, name_col):
    """处理单个目标：下载 -> 分析 -> 绘图"""
    
    # 1. 获取基本信息
    ra = float(row[ra_col])
    dec = float(row[dec_col])
    
    # 获取名字并清理
    raw_name = row[name_col]
    safe_name = sanitize_filename(raw_name)
    
    # 定义文件路径 (直接用 Name 命名)
    csv_filename = f"{safe_name}.csv"
    csv_filepath = os.path.join(DATA_DIR, csv_filename)
    
    print(f"处理: {safe_name} (RA={ra:.5f}, DEC={dec:.5f}) ... ", end="", flush=True)
    
    # 2. 下载数据 (如果本地没有)
    if not os.path.exists(csv_filepath):
        success = download_ztf(ra, dec, csv_filepath)
        if not success: 
            print("下载失败")
            return
    
    # 检查文件是否为空 (ZTF 如果没找到源，有时会下载一个很小的空文件)
    if os.path.getsize(csv_filepath) < 100:
        print("无数据 (文件过小)")
        # os.remove(csv_filepath) # 可选：删除空文件
        return

    # 3. 读取数据
    try:
        data = pd.read_csv(csv_filepath)
        if data.empty or 'filtercode' not in data.columns:
            print("数据格式错误")
            return
    except:
        print("文件损坏")
        return

    # 4. 数据清洗
    # 过滤 bad flags 和 波段
    gdata = data[(data.filtercode == 'zg') & (data.catflags == 0)].sort_values(by='mjd').reset_index(drop=True)
    rdata = data[(data.filtercode == 'zr') & (data.catflags == 0)].sort_values(by='mjd').reset_index(drop=True)
    idata = data[(data.filtercode == 'zi') & (data.catflags == 0)].sort_values(by='mjd').reset_index(drop=True)

    if len(gdata) < 10 and len(rdata) < 10:
        print("点数不足 (<10)")
        return

    # 5. 数据归一化与周期分析
    try:
        # 寻找时间参考点 tref0
        all_hjd = []
        if len(gdata) > 0: all_hjd.extend(gdata.hjd.values)
        if len(rdata) > 0: all_hjd.extend(rdata.hjd.values)
        if len(all_hjd) == 0: return
        tref0 = min(all_hjd)

        # G 波段处理
        if len(gdata) > 5:
            t_g = np.array(gdata.hjd) - tref0
            y_g = (np.array(gdata.mag) - min(gdata.mag)) / (max(gdata.mag) - min(gdata.mag))
            dy_g = np.array(gdata.magerr)
            period_g, freq_g, per_g, _, _ = AOV(t_g, y_g, dy_g)
        else:
            t_g, y_g, dy_g, period_g = [], [], [], None

        # R 波段处理
        if len(rdata) > 5:
            t_r = np.array(rdata.hjd) - tref0
            y_r = (np.array(rdata.mag) - min(rdata.mag)) / (max(rdata.mag) - min(rdata.mag))
            dy_r = np.array(rdata.magerr)
            period_r, freq_r, per_r, _, _ = AOV(t_r, y_r, dy_r)
        else:
            t_r, y_r, dy_r, period_r = [], [], [], None
        
        # I 波段数据准备
        if len(idata) > 0:
            t_i = np.array(idata.hjd) - tref0
            y_i = (np.array(idata.mag) - min(idata.mag)) / (max(idata.mag) - min(idata.mag))
            dy_i = np.array(idata.magerr)
        else:
            t_i, y_i, dy_i = [], [], []

        # 确定最佳周期 (优先 R, 其次 G)
        period = period_r if period_r else period_g
        
        if period is None:
            print("无法确定周期")
            return

        print(f"周期: {period*24*60:.2f} min")

    except Exception as e:
        print(f"计算出错: {e}")
        return

    # 6. 计算相位 (Fold)
    periods_g = fold(gdata.hjd.values - tref0, period) if len(gdata) > 0 else []
    periods_r = fold(rdata.hjd.values - tref0, period) if len(rdata) > 0 else []
    periods_i = fold(idata.hjd.values - tref0, period) if len(idata) > 0 else []

    # ================= 绘图 =================
    figd = plt.figure('d', figsize=(14, 16))
    gsd = GridSpec(6, 1, height_ratios=[4.0, 1.5, 4.0, 1.0, 2.0, 2.0], hspace=0)
    
    ax0 = figd.add_subplot(gsd[0]) # 原始光变
    ax2 = figd.add_subplot(gsd[2]) # 折叠光变
    ax4 = figd.add_subplot(gsd[4]) # g 波段周期图
    ax5 = figd.add_subplot(gsd[5]) # r 波段周期图

    # Plot Raw
    if len(gdata) > 0: ax0.errorbar(gdata.mjd, gdata.mag, yerr=gdata.magerr, c='g', label='g', marker='x', ls='none')
    if len(rdata) > 0: ax0.errorbar(rdata.mjd, rdata.mag, yerr=rdata.magerr, c='r', label='r', marker='o', ls='none')
    if len(idata) > 0: ax0.errorbar(idata.mjd, idata.mag, yerr=idata.magerr, c='b', label='i', marker='^', ls='none')

    ax0.set_xlabel('MJD', fontsize=24)
    ax0.set_ylabel('Mag', fontsize=24)
    ax0.tick_params(labelsize=24)
    ax0.invert_yaxis()
    ax0.set_title(f'{safe_name} (RA={ra:.5f} Dec={dec:.5f})', fontsize=20)
    
    # Plot Folded
    for i in range(3):
        if len(gdata) > 0: ax2.errorbar(periods_g + float(i), gdata.mag, yerr=gdata.magerr, c='g', marker='x', ls='none')
        if len(rdata) > 0: ax2.errorbar(periods_r + float(i), rdata.mag, yerr=rdata.magerr, c='r', marker='o', ls='none')
        if len(idata) > 0: ax2.errorbar(periods_i + float(i), idata.mag, yerr=idata.magerr, c='b', marker='^', ls='none')

    ax2.set_xlim(-0.05, 2.05)
    ax2.set_title(f'Folded on Period = {period*24*60:.3f} min', fontsize=18)
    ax2.set_xlabel('Phase', fontsize=24)
    ax2.set_ylabel('Mag', fontsize=24)
    ax2.invert_yaxis()
    ax2.tick_params(labelsize=24)
    # Fix Y-limit inversion check
    ymin, ymax = ax2.get_ylim()
    if ymin < ymax: ax2.set_ylim(ymax, ymin)

    # Plot Periodograms
    if period_g:
        ax4.plot(freq_g, per_g, ls='-', c='g')
    ax4.set_ylabel('Power (g)', fontsize=20)
    ax4.set_xlim(0, 100)
    ax4.tick_params(labelbottom=False, labelsize=24)

    if period_r:
        ax5.plot(freq_r, per_r, ls='-', c='r')
    ax5.set_ylabel('Power (r)', fontsize=20)
    ax5.set_xlim(0, 100)
    ax5.set_xlabel('Frequency [1/day]', fontsize=24)
    ax5.tick_params(labelsize=24)

    # Save
    plot_filename = f"{safe_name}.png"
    save_path = os.path.join(OUTPUT_PLOT_DIR, plot_filename)
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()

# ================= 主程序 =================
def main():
    print(f"读取文件: {INPUT_FILE}")
    if not os.path.exists(INPUT_FILE):
        print("文件不存在")
        return

    df = pd.read_csv(INPUT_FILE)
    
    # 自动识别列名
    cols = {c.lower(): c for c in df.columns}
    ra_col = cols.get('ra') or cols.get('ra_deg') or cols.get('alpha')
    dec_col = cols.get('dec') or cols.get('dec_deg') or cols.get('delta')
    # 寻找名字列
    name_col = cols.get('name') or cols.get('identifier') or cols.get('source') or cols.get('objname')

    if not ra_col or not dec_col:
        print("无法识别 RA 或 Dec 列名")
        return
    
    if not name_col:
        print("无法识别 Name/Identifier 列名，将使用行号命名")
        # 如果没有名字，临时创建一个
        df['Auto_Name'] = [f"Obj_{i}" for i in range(len(df))]
        name_col = 'Auto_Name'

    print(f"使用列: Name='{name_col}', RA='{ra_col}', Dec='{dec_col}'")
    print(f"开始处理 {len(df)} 个目标...")

    for index, row in df.iterrows():
        try:
            process_target(row, ra_col, dec_col, name_col)
        except KeyboardInterrupt:
            print("\n用户中断")
            break
        except Exception as e:
            print(f"\n[Error] 未知错误: {e}")

if __name__ == "__main__":
    main()

In [ ]:
画和罗图，要求线simbad出他gaia的信息，然后再结合TAP_1632913590914.csv文件背景图画出和罗图

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def plot_hr_diagram_marked():
    # ================= 配置区 =================
    # 背景文件 (Gaia 随机样本)
    bg_file = "TAP_1632913590914.csv"
    # 目标文件 (你的列表)
    target_file = "apjad7500t1_mrt.csv"
    
    # 需要特殊标记的星名 (必须与 Candidate 列完全一致)
    special_targets = {
        'UPK_13-c2': {'marker': '*', 'color': 'red', 's': 300, 'label': 'UPK_13-c2'},
        'Alessi_12-c1': {'marker': 's', 'color': 'blue', 's': 150, 'label': 'Alessi_12-c1'}
    }

    # ================= 数据处理函数 =================
    def process_data(df, is_target=False):
        # 1. 统一列名 (转小写方便查找)
        # 创建一个列名映射字典，方便后续引用
        cols = {c.lower(): c for c in df.columns}
        
        # 2. 智能查找关键列
        # 视差 (优先找 plx, 其次 parallax)
        col_plx = cols.get('plx') or cols.get('parallax')
        
        # G波段星等
        col_g = cols.get('gmag') or cols.get('phot_g_mean_mag')
        
        # 颜色计算 (BP - RP)
        # 如果有直接的 bp_rp 列最好，没有则尝试用 bpmag - rpmag 计算
        col_bp_rp = cols.get('bp_rp') or cols.get('bp-rp')
        col_bp = cols.get('bpmag') or cols.get('phot_bp_mean_mag')
        col_rp = cols.get('rpmag') or cols.get('phot_rp_mean_mag')
        
        col_name = None
        if is_target:
            # 查找名字列 (Candidate)
            col_name = cols.get('candidate')

        # 3. 检查必要列
        if not col_plx or not col_g:
            print(f"⚠️  缺少必要列 (Plx 或 Gmag)，无法计算绝对星等。")
            return None, None, None, None

        # 4. 过滤有效数据 (视差必须 > 0 才能取对数)
        mask = (df[col_plx] > 0) & (df[col_plx].notnull()) & (df[col_g].notnull())
        df_clean = df[mask].copy()
        
        # 5. 计算绝对星等 M_G
        # 公式: M = m + 5 - 5 * log10(distance_pc)
        # distance_pc = 1000 / plx_mas
        # 合并公式: M = m + 5 * log10(plx/1000) + 5
        df_clean['M_G'] = df_clean[col_g] + 5 * np.log10(df_clean[col_plx] / 1000.0) + 5
        
        # 6. 计算或获取颜色 (BP-RP)
        if col_bp_rp and col_bp_rp in df_clean.columns:
            df_clean['Color_Index'] = df_clean[col_bp_rp]
        elif col_bp and col_rp:
            df_clean['Color_Index'] = df_clean[col_bp] - df_clean[col_rp]
        else:
            print("⚠️  无法找到 BP/RP 数据计算颜色。")
            return None, None, None, None
            
        return df_clean, 'Color_Index', 'M_G', col_name

    # ================= 读取数据 =================
    print("正在读取背景数据...")
    try:
        df_bg = pd.read_csv(bg_file)
        bg_data, bg_x, bg_y, _ = process_data(df_bg, is_target=False)
    except Exception as e:
        print(f"❌ 背景文件读取失败: {e}")
        bg_data = None

    print("正在读取目标数据...")
    try:
        # 尝试读取目标文件，鉴于你提供的内容可能是制表符分隔，这里加上 sep='\t' 尝试
        # 如果你的文件是逗号分隔，把 sep='\t' 去掉即可
        try:
            df_tar = pd.read_csv(target_file)
            # 如果读出来只有1列，说明分隔符不对，尝试制表符
            if df_tar.shape[1] < 2:
                df_tar = pd.read_csv(target_file, sep='\t')
        except:
             df_tar = pd.read_csv(target_file, sep='\t')
             
        tar_data, tar_x, tar_y, tar_name_col = process_data(df_tar, is_target=True)
    except Exception as e:
        print(f"❌ 目标文件读取失败: {e}")
        return

    if tar_data is None:
        print("❌ 目标数据处理后为空，请检查列名。")
        return

    # ================= 开始绘图 =================
    fig, ax = plt.subplots(figsize=(10, 10))

    # 1. 画背景 (灰色)
    if bg_data is not None:
        ax.scatter(bg_data[bg_x], bg_data[bg_y], 
                   s=1, c='gray', alpha=0.15, label='Gaia Background', rasterized=True)

    # 2. 画所有目标成员星 (黑色空心圆)
    ax.scatter(tar_data[tar_x], tar_data[tar_y], 
               s=40, facecolors='none', edgecolors='k', alpha=0.6, label='Cluster Members')

    # 3. 特殊标记 UPK_13-c2 和 Alessi_12-c1
    print("\n--- 标记特殊目标 ---")
    for target_name, style in special_targets.items():
        # 查找对应的行
        row = tar_data[tar_data[tar_name_col] == target_name]
        
        if not row.empty:
            x_val = row[tar_x].values[0]
            y_val = row[tar_y].values[0]
            
            # 绘制标记
            ax.scatter(x_val, y_val, 
                       s=style['s'], c=style['color'], marker=style['marker'], 
                       edgecolors='k', zorder=10, label=style['label'])
            
            # 添加文字标签
            ax.text(x_val + 0.1, y_val, target_name, fontsize=12, fontweight='bold', color=style['color'])
            
            print(f"✅ 已标记: {target_name} (Color={x_val:.3f}, AbsMag={y_val:.3f})")
        else:
            print(f"⚠️  未在数据中找到: {target_name} (请检查csv中的 Candidate 列名拼写)")

    # ================= 装饰图表 =================
    ax.invert_yaxis() # 星等越小越亮，反转Y轴
    
    ax.set_xlabel('Color ($G_{BP} - G_{RP}$)', fontsize=14)
    ax.set_ylabel('Absolute Magnitude ($M_G$)', fontsize=14)
    ax.set_title('Gaia Color-Magnitude Diagram', fontsize=16, pad=15)
    
    # 调整显示范围 (经典赫罗图范围)
    ax.set_xlim(-0.5, 4.0) # 根据数据情况微调
    ax.set_ylim(15, -2)    # 根据数据情况微调
    
    # 图例
    ax.legend(loc='upper left', fontsize=12, frameon=True)
    ax.grid(True, linestyle='--', alpha=0.4)
    
    # 保存
    out_file = 'CMD_Marked_Targets.png'
    plt.savefig(out_file, dpi=300, bbox_inches='tight')
    print(f"\n✅ 绘图完成！已保存为 {out_file}")
    plt.show()

if __name__ == "__main__":
    plot_hr_diagram_marked()